# Rubik's Cube Solver: training notebook (Google Colab)

**What this notebook does**
1. Trains a CNN **from scratch** that reads one cube face (96x96 image) and outputs the colour of all 9 stickers.
2. Evaluates it honestly: on faces whose colours come from a lighting condition the model never saw, and at **cube level** (are all 54 stickers right, and does the Kociemba solver then solve it?).
3. Compares it with two classical baselines.
4. Saves `face_cnn.keras` for the Streamlit app.

**Solving itself is not learned.** The Kociemba two-phase algorithm solves the cube; the deep learning part is the vision model.

**Set the runtime to T4 GPU** (Runtime > Change runtime type) before running.

## Step 1: Install and download the colour dataset

In [ ]:
!pip install -q kociemba
!git clone -q https://github.com/eyeeco/Rubick-s-Cube-Dataset

## Step 2: Write the helper modules
These three files are also shipped with the Streamlit app (`cube_utils.py` is shared by both).

In [ ]:
%%writefile cube_utils.py
"""cube_utils.py - everything in the Rubik's cube solver that is NOT the neural network.

Face order and facelet layout follow the Kociemba convention:
    U R F D L B, each face = 9 stickers row by row, as seen in the unfolded net

                 U
               L F R B
                 D
"""
import numpy as np
import cv2
from collections import Counter
from PIL import Image, ImageDraw

COLORS = ["white", "yellow", "red", "orange", "blue", "green"]       # class index order of the CNN
ORDER = ["U", "R", "F", "D", "L", "B"]
SOLVED = "".join(L * 9 for L in ORDER)
STD_SCHEME = {"U": "white", "R": "red", "F": "green", "D": "yellow", "L": "orange", "B": "blue"}
RGB = {"white": (240, 240, 240), "yellow": (250, 215, 40), "red": (200, 30, 40),
       "orange": (250, 130, 20), "blue": (30, 80, 200), "green": (30, 160, 70)}

# --------------------------------------------------------------------------- simulator
def _facelets():
    out = []
    for L in ORDER:
        for r in range(3):
            for c in range(3):
                if L == "U":   pos, n = (c - 1, 1, r - 1), (0, 1, 0)
                elif L == "R": pos, n = (1, 1 - r, 1 - c), (1, 0, 0)
                elif L == "F": pos, n = (c - 1, 1 - r, 1), (0, 0, 1)
                elif L == "D": pos, n = (c - 1, -1, 1 - r), (0, -1, 0)
                elif L == "L": pos, n = (-1, 1 - r, c - 1), (-1, 0, 0)
                else:          pos, n = (1 - c, 1 - r, -1), (0, 0, -1)
                out.append((pos, n))
    return out

_FL = _facelets()
_INDEX = {fl: i for i, fl in enumerate(_FL)}
_AXIS = {"U": (0, 1, 0), "R": (1, 0, 0), "F": (0, 0, 1), "D": (0, -1, 0), "L": (-1, 0, 0), "B": (0, 0, -1)}

def _rot_cw(v, a):
    """Rotate vector v by +90 degrees about the outward axis a. With this cube coordinate system
    (x right, y up, z towards the viewer) that is the clockwise quarter turn seen from outside."""
    v, a = np.array(v), np.array(a)
    return tuple(int(x) for x in (np.cross(a, v) + a * np.dot(a, v)))

def _perm(face):
    a = _AXIS[face]
    perm = list(range(54))
    for i, (pos, n) in enumerate(_FL):
        if np.dot(pos, a) == 1:
            perm[_INDEX[(_rot_cw(pos, a), _rot_cw(n, a))]] = i     # sticker i moves to this slot
    return perm

_PERMS = {f: _perm(f) for f in ORDER}

def apply_move(state, move):
    """state: string/list of 54 letters. move: 'R', "R'", 'R2'."""
    face, turns = move[0], {"": 1, "'": 3, "2": 2}[move[1:]]
    s = list(state)
    for _ in range(turns):
        s = _once(s, face)
    return "".join(s)

def _once(s, face):
    new = list(s)
    for src, dst in enumerate(_PERMS[face]):
        new[dst] = s[src]
    return new

def apply_moves(state, moves):
    for m in (moves.split() if isinstance(moves, str) else moves):
        state = apply_move(state, m)
    return state

def random_scramble(rng, n=25):
    moves, last = [], None
    for _ in range(n):
        f = rng.choice([x for x in ORDER if x != last]); last = f
        moves.append(f + rng.choice(["", "'", "2"]))
    return moves

# --------------------------------------------------------------------------- face detection
def _order_corners(pts):
    pts = pts.reshape(4, 2).astype("float32")
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.array([pts[s.argmin()], pts[d.argmin()], pts[s.argmax()], pts[d.argmax()]], dtype="float32")

def extract_face(bgr, size=96):
    """Find the biggest square-ish 4-corner shape (the cube face) and flatten it.
    Falls back to the centre square of the photo if nothing is found. Returns (BGR image, found)."""
    gray = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(gray, 40, 120), np.ones((3, 3), np.uint8))
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best, best_area = None, 0.08 * bgr.shape[0] * bgr.shape[1]
    for c in cnts:
        hull = cv2.convexHull(c)
        approx = cv2.approxPolyDP(hull, 0.03 * cv2.arcLength(hull, True), True)
        area = cv2.contourArea(approx)
        if len(approx) == 4 and area > best_area:
            q = _order_corners(approx)
            w1, w2 = np.linalg.norm(q[0] - q[1]), np.linalg.norm(q[0] - q[3])
            if 0.6 < w1 / max(w2, 1) < 1.6:
                best, best_area = q, area
    if best is None:
        h, w = bgr.shape[:2]; m = min(h, w)
        y0, x0 = (h - m) // 2, (w - m) // 2
        return cv2.resize(bgr[y0:y0 + m, x0:x0 + m], (size, size)), False
    dst = np.float32([[0, 0], [size, 0], [size, size], [0, size]])
    return cv2.warpPerspective(bgr, cv2.getPerspectiveTransform(best, dst), (size, size)), True

def sticker_lab(face_bgr):
    """Median Lab colour of the centre third of each of the 9 stickers -> (9, 3) float array."""
    n = face_bgr.shape[0] // 3
    m = max(1, n // 3)
    out = []
    for r in range(3):
        for c in range(3):
            patch = face_bgr[r * n + m:(r + 1) * n - m, c * n + m:(c + 1) * n - m].reshape(-1, 3)
            med = np.median(patch, axis=0).astype(np.uint8).reshape(1, 1, 3)
            out.append(cv2.cvtColor(med, cv2.COLOR_BGR2LAB)[0, 0].astype(float))
    return np.array(out)

def sticker_hsv(face_bgr):
    n = face_bgr.shape[0] // 3
    m = max(1, n // 3)
    out = []
    for r in range(3):
        for c in range(3):
            patch = face_bgr[r * n + m:(r + 1) * n - m, c * n + m:(c + 1) * n - m].reshape(-1, 3)
            med = np.median(patch, axis=0).astype(np.uint8).reshape(1, 1, 3)
            out.append(cv2.cvtColor(med, cv2.COLOR_BGR2HSV)[0, 0].astype(float))
    return np.array(out)

# --------------------------------------------------------------------------- colour decoding
def constrained_decode(probs):
    """probs: (54, 6) CNN probabilities for the stickers in U,R,F,D,L,B order.
    A real cube has exactly 9 stickers of each colour and 6 different centres, so pick the most
    likely assignment that respects that (linear assignment). Returns 54 colour indices."""
    from scipy.optimize import linear_sum_assignment
    logp = np.log(np.clip(probs, 1e-9, 1.0))
    centres = [4 + 9 * k for k in range(6)]
    others = [i for i in range(54) if i not in centres]
    out = np.zeros(54, dtype=int)
    r, c = linear_sum_assignment(-logp[centres])                 # 6 centres -> 6 different colours
    out[np.array(centres)[r]] = c
    cost = -np.repeat(logp[others], 8, axis=1)                   # 48 stickers -> 6 colours x 8 slots
    r, c = linear_sum_assignment(cost)
    out[np.array(others)[r]] = c // 8
    return out

def centre_nearest(lab6):
    """Classical baseline, no training: lab6 is (6 faces, 9, 3). Every sticker gets the colour of the
    nearest face centre. Centres are named by matching them to the standard palette."""
    from scipy.optimize import linear_sum_assignment
    w = np.array([0.5, 1.0, 1.0])
    centres = lab6[:, 4]
    flat = lab6.reshape(54, 3)
    d = (((flat[:, None, :] - centres[None]) * w) ** 2).sum(-1)
    face_of = d.argmin(1)
    std = np.array([cv2.cvtColor(np.uint8([[RGB[c][::-1]]]), cv2.COLOR_BGR2LAB)[0, 0] for c in COLORS], float)
    dc = (((centres[:, None, :] - std[None]) * w) ** 2).sum(-1)
    r, c = linear_sum_assignment(dc)
    name_of_face = dict(zip(r, c))
    return np.array([name_of_face[f] for f in face_of])

# --------------------------------------------------------------------------- state and solver
def build_state(face_colors):
    """face_colors: dict letter -> list of 9 colour names (row by row). Returns the 54-letter string."""
    centre_to_letter = {face_colors[L][4]: L for L in ORDER}
    if len(centre_to_letter) != 6:
        raise ValueError("The 6 centre stickers must be 6 different colours, but they are "
                         + ", ".join(face_colors[L][4] for L in ORDER) + ".")
    counts = Counter(c for L in ORDER for c in face_colors[L])
    if any(counts[c] != 9 for c in COLORS):
        many = [f"{c} ({counts[c]})" for c in COLORS if counts[c] > 9]
        few = [f"{c} ({counts[c]})" for c in COLORS if counts[c] < 9]
        raise ValueError("Each colour must appear exactly 9 times. Too many: " + (", ".join(many) or "none")
                         + ". Too few: " + (", ".join(few) or "none") + ". Fix the wrong stickers in step 3.")
    return "".join(centre_to_letter[c] for L in ORDER for c in face_colors[L])

def solve(state):
    import kociemba
    if state == SOLVED:
        return ""
    try:
        return kociemba.solve(state)
    except ValueError:
        raise ValueError("This cube is impossible, so a sticker is probably misread. "
                         "Fix the wrong sticker and try again.")

# --------------------------------------------------------------------------- drawing
_NET_POS = {"U": (1, 0), "L": (0, 1), "F": (1, 1), "R": (2, 1), "B": (3, 1), "D": (1, 2)}

def draw_net(colors54, cell=26, gap=6):
    """colors54: 54 colour names in U,R,F,D,L,B order. Returns a PIL image of the unfolded cube."""
    W, H = 4 * (3 * cell + gap), 3 * (3 * cell + gap)
    img = Image.new("RGB", (W, H), (255, 255, 255))
    d = ImageDraw.Draw(img)
    for k, L in enumerate(ORDER):
        bx, by = _NET_POS[L]
        for i in range(9):
            r, c = divmod(i, 3)
            x = bx * (3 * cell + gap) + c * cell
            y = by * (3 * cell + gap) + r * cell
            d.rectangle([x, y, x + cell - 2, y + cell - 2], fill=RGB[colors54[k * 9 + i]], outline=(40, 40, 40))
    return img

def state_to_colors(state, letter_color):
    return [letter_color[ch] for ch in state]


In [ ]:
%%writefile synth.py
"""Synthetic Rubik's cube face generator.
Sticker colours are sampled from the real HSV dataset, so they carry real lighting variation.
Each image mimics what extract_face() returns: a flattened 96x96 face with small detection errors."""
import random
import numpy as np
import cv2
from cube_utils import COLORS

CSV_LABELS = {0: "green", 1: "white", 2: "yellow", 3: "red", 4: "orange", 5: "blue"}

def load_color_rows(csv_files):
    """csv_files: list of paths. Returns {colour name: (n, 3) array of OpenCV-scale H, S, V}."""
    import pandas as pd
    rows = {c: [] for c in COLORS}
    for f in csv_files:
        d = pd.read_csv(f, header=None, names=["h", "s", "v", "label"])
        for k, name in CSV_LABELS.items():
            rows[name].append(d[d.label == k][["h", "s", "v"]].values)
    return {c: np.concatenate(v).astype("float32") for c, v in rows.items()}

def _sticker_bgr(rows, name, rng, shift):
    h, s, v = rows[name][rng.randrange(len(rows[name]))]
    h = (h + shift[0]) % 180
    s = float(np.clip(s * shift[1], 0, 255)); v = float(np.clip(v * shift[2] * rng.uniform(0.95, 1.05), 0, 255))
    return cv2.cvtColor(np.uint8([[[h, s, v]]]), cv2.COLOR_HSV2BGR)[0, 0].tolist()

def render_face(colors9, rows, rng, out=96):
    """colors9: 9 colour names, row by row. Returns an RGB uint8 image (out, out, 3)."""
    S = 192
    img = np.full((S, S, 3), rng.randint(10, 50), np.uint8)                  # black plastic
    border = S * rng.uniform(0.02, 0.06)
    cell = (S - 2 * border) / 3
    gap = cell * rng.uniform(0.06, 0.14)
    shift = (rng.uniform(-4, 4), rng.uniform(0.85, 1.12), rng.uniform(0.85, 1.12))   # camera / cube differences
    for i, name in enumerate(colors9):
        r, c = divmod(i, 3)
        x0, y0 = int(border + c * cell + gap), int(border + r * cell + gap)
        x1, y1 = int(border + (c + 1) * cell - gap), int(border + (r + 1) * cell - gap)
        cv2.rectangle(img, (x0, y0), (x1, y1), _sticker_bgr(rows, name, rng, shift), -1)
    img = img.astype(np.float32)
    yy, xx = np.mgrid[0:S, 0:S] / S                                          # lighting gradient
    img *= (1 + rng.uniform(-.25, .25) * (xx - .5) + rng.uniform(-.25, .25) * (yy - .5))[..., None]
    if rng.random() < 0.3:                                                   # glare
        cx, cy, rad = rng.uniform(0, 1), rng.uniform(0, 1), rng.uniform(0.1, 0.35)
        img += (np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * rad ** 2)) * rng.uniform(40, 130))[..., None]
    if rng.random() < 0.2:                                                   # shadow band
        a = rng.uniform(0, 1); img *= np.where(np.abs(xx - a) < 0.15, rng.uniform(0.6, 0.85), 1.0)[..., None]
    img = np.clip(img, 0, 255).astype(np.uint8)
    j = out * 0.04                                                           # imperfect face detection
    src = np.float32([[0, 0], [S, 0], [S, S], [0, S]])
    dst = np.float32([[0, 0], [out, 0], [out, out], [0, out]]) + np.float32(
        [[rng.uniform(-j, j), rng.uniform(-j, j)] for _ in range(4)])
    img = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (out, out), borderMode=cv2.BORDER_REPLICATE)
    if rng.random() < 0.5:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.4, 1.3))
    img = np.clip(img.astype(np.float32) + np.random.normal(0, rng.uniform(0, 6), img.shape), 0, 255).astype(np.uint8)
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, rng.randint(40, 95)])
    return cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)

def make_dataset(n, rows, seed, out=96):
    """n random faces (any colour on any sticker). Returns X (n, out, out, 3) uint8, y (n, 3, 3) class ids."""
    rng = random.Random(seed); np.random.seed(seed)
    X = np.empty((n, out, out, 3), np.uint8); y = np.empty((n, 9), np.int64)
    for k in range(n):
        ids = [rng.randrange(6) for _ in range(9)]
        X[k] = render_face([COLORS[i] for i in ids], rows, rng, out); y[k] = ids
    return X, y.reshape(n, 3, 3)


In [ ]:
%%writefile train_eval.py
"""Model + evaluation helpers used by the training notebook."""
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
from cube_utils import *
from synth import render_face

def build_face_cnn(size=96, widths=(32, 64, 128, 256)):
    """Fully convolutional: the 3x3 output grid = one prediction per sticker (6 colour classes)."""
    def block(x, f):
        for _ in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization(momentum=0.9)(x)
            x = layers.Activation("relu")(x)
        return layers.MaxPooling2D()(x)
    inp = keras.Input((size, size, 3))
    x = layers.Rescaling(1. / 255)(inp)
    for f in widths:                                    # 96 -> 48 -> 24 -> 12 -> 6
        x = block(x, f)
    x = layers.AveragePooling2D(2)(x)                  # 6x6 -> 3x3: one cell per sticker
    x = layers.Conv2D(128, 1, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Conv2D(6, 1, activation="softmax")(x)  # (3, 3, 6)
    return keras.Model(inp, out)

def face_probs(model, faces_rgb):
    """faces_rgb: (n, 96, 96, 3) uint8 -> probabilities (n*9, 6), stickers row by row."""
    return model.predict(faces_rgb, verbose=0).reshape(-1, 6)

# ------------------------------------------------------------------ face-level metrics
def face_metrics(pred_idx, y):
    """pred_idx, y: (n, 9). Returns (sticker accuracy, full-face accuracy)."""
    ok = pred_idx == y
    return ok.mean(), ok.all(axis=1).mean()

# ------------------------------------------------------------------ cube-level evaluation
def cube_eval(readers, rows, n_cubes=100, seed=7, scramble_len=25):
    """readers: dict name -> function(faces_rgb (6,96,96,3), faces_bgr (6,96,96,3)) -> 54 colour indices.
    Builds random scrambled cubes, renders the 6 faces, lets each reader read them and checks:
      sticker accuracy, exact cube state (54/54), and whether Kociemba then returns a correct solution."""
    rng = random.Random(seed); np.random.seed(seed)
    res = {k: dict(sticker=0, exact=0, solved=0) for k in readers}
    for _ in range(n_cubes):
        state = apply_moves(SOLVED, random_scramble(rng, scramble_len))
        truth = [STD_SCHEME[ch] for ch in state]
        faces = np.stack([render_face(truth[f * 9:(f + 1) * 9], rows, rng) for f in range(6)])
        bgr = faces[..., ::-1].copy()
        t_idx = np.array([COLORS.index(c) for c in truth])
        for name, fn in readers.items():
            p = np.asarray(fn(faces, bgr))
            res[name]["sticker"] += (p == t_idx).mean()
            try:
                st = build_state({L: [COLORS[i] for i in p[k * 9:(k + 1) * 9]] for k, L in enumerate(ORDER)})
                if st == state:
                    res[name]["exact"] += 1
                    res[name]["solved"] += apply_moves(state, solve(state)) == SOLVED
            except ValueError:
                pass
    return {k: dict(sticker_acc=v["sticker"] / n_cubes, exact_cube=v["exact"] / n_cubes,
                    solved=v["solved"] / n_cubes) for k, v in res.items()}


## Step 3: Colour data and the unseen-lighting split
The dataset has 6264 stickers measured under 5 lighting conditions (H, S, V values with a colour label).
Training faces use colours from 4 conditions; **test faces use the 5th condition only**, so the test measures generalisation to new lighting.

In [ ]:
import glob, random, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from cube_utils import *
from synth import *
from train_eval import *

files = sorted(glob.glob("Rubick-s-Cube-Dataset/*_labels.csv"))
HELD_OUT = "cube2_up_light"
rows_test  = load_color_rows([f for f in files if HELD_OUT in f])
rows_train = load_color_rows([f for f in files if HELD_OUT not in f])
print({k: len(v) for k, v in rows_train.items()}, "training colour samples")
print({k: len(v) for k, v in rows_test.items()}, "held-out colour samples")

## Step 4: Look at the synthetic faces
Each image imitates what the face detector returns: lighting gradient, glare, blur, noise, JPEG artefacts, small detection errors.

In [ ]:
X, y = make_dataset(24, rows_train, seed=0)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, img, lab in zip(axes.ravel(), X, y):
    ax.imshow(img); ax.axis("off")
    ax.set_title("".join(COLORS[i][0].upper() for i in lab.ravel()), fontsize=7)   # W Y R O B G per sticker
plt.tight_layout(); plt.show()

## Step 5: Generate the datasets (about 3 to 5 minutes)

In [ ]:
N_TRAIN, N_VAL, N_TEST = 40000, 4000, 4000
X_train, y_train = make_dataset(N_TRAIN, rows_train, seed=1)
X_val,   y_val   = make_dataset(N_VAL,   rows_train, seed=2)
X_test,  y_test  = make_dataset(N_TEST,  rows_test,  seed=3)      # unseen lighting condition
print(X_train.shape, y_train.shape, X_test.shape)

## Step 6: Build and train the CNN (from scratch, no pretrained weights)

In [ ]:
model = build_face_cnn()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("face_cnn.keras", save_best_only=True, monitor="val_loss"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss"),
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=25, batch_size=128, callbacks=callbacks)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Sticker accuracy"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend(); plt.show()

## Step 7: Face-level test on the unseen lighting condition

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred = model.predict(X_test, verbose=0).argmax(-1).reshape(-1, 9)
stk, face = face_metrics(pred, y_test.reshape(-1, 9))
print(f"Sticker accuracy {stk*100:.2f}% | full-face accuracy (9 of 9 right) {face*100:.2f}%")

ConfusionMatrixDisplay(confusion_matrix(y_test.reshape(-1), pred.reshape(-1)),
                       display_labels=COLORS).plot(cmap="Blues", xticks_rotation=45)
plt.show()

## Step 8: Cube-level test and comparison with baselines
One wrong sticker out of 54 makes the cube unsolvable, so this is the number that matters.
For each random scrambled cube, the 6 faces are rendered with the **held-out lighting**, read by each method, and the result is checked:
* `sticker_acc`: share of the 54 stickers read correctly
* `exact_cube`: all 54 correct
* `solved`: exact state AND the Kociemba solution really returns the cube to solved

Methods: the CNN (plain), the CNN with **constrained decoding** (a cube has exactly 9 stickers per colour), a Random Forest on HSV values, and a no-training method that matches every sticker to the nearest face centre.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

def hsv_feats(faces_bgr): return np.concatenate([sticker_hsv(f) for f in faces_bgr])
rf = RandomForestClassifier(200, random_state=0, n_jobs=-1).fit(
        hsv_feats(X_train[:4000, :, :, ::-1].copy()), y_train[:4000].reshape(-1))

readers = {
    "CNN (plain)":            lambda fr, fb: face_probs(model, fr).argmax(1),
    "CNN + constrained":      lambda fr, fb: constrained_decode(face_probs(model, fr)),
    "HSV random forest":      lambda fr, fb: rf.predict(hsv_feats(fb)),
    "Centre-nearest (no ML)": lambda fr, fb: centre_nearest(np.stack([sticker_lab(f) for f in fb])),
}
res = cube_eval(readers, rows_test, n_cubes=200)
pd.DataFrame(res).T.style.format("{:.1%}")

## Step 9: Save the model
Download `face_cnn.keras` and put it next to `app.py`.

In [ ]:
model.save("face_cnn.keras")
from google.colab import files
files.download("face_cnn.keras")

## Step 10: Test on YOUR cube (the real test)
Synthetic results are only a guide. Your cube and camera may look different, so measure on real photos.

1. Photograph the 6 faces of your cube (see the scan order in the app: F, R, B, L with U on top, then U, then D).
2. Name the files `U.jpg R.jpg F.jpg D.jpg L.jpg B.jpg` and upload them below.
3. Fill in the true colours of every face, row by row, using the letters `w y r o b g`.

In [ ]:
from google.colab import files
import cv2
up = files.upload()

LETTER = {"w": 0, "y": 1, "r": 2, "o": 3, "b": 4, "g": 5}
TRUTH = {                      # edit these to match YOUR cube, 9 letters per face, row by row
    "U": "wwwwwwwww", "R": "rrrrrrrrr", "F": "ggggggggg",
    "D": "yyyyyyyyy", "L": "ooooooooo", "B": "bbbbbbbbb",
}

faces_bgr = []
for L in ORDER:
    img = cv2.imread(f"{L}.jpg")
    face, found = extract_face(img, 96)
    faces_bgr.append(face); print(L, "face outline found:", found)
faces_bgr = np.stack(faces_bgr); faces_rgb = faces_bgr[..., ::-1].copy()
truth_idx = np.array([LETTER[ch] for L in ORDER for ch in TRUTH[L]])

probs = face_probs(model, faces_rgb)
for name, p in {"CNN (plain)": probs.argmax(1), "CNN + constrained": constrained_decode(probs),
                "Centre-nearest": centre_nearest(np.stack([sticker_lab(f) for f in faces_bgr]))}.items():
    print(f"{name:18s} stickers correct: {(p == truth_idx).sum()}/54")

final = constrained_decode(probs)
colors = {L: [COLORS[i] for i in final[k * 9:(k + 1) * 9]] for k, L in enumerate(ORDER)}
state = build_state(colors)
moves = solve(state)
print("Solution:", moves or "already solved")

## Step 11 (optional): Fine-tune on your own real faces
If the real-photo accuracy is low, add real labelled faces. Use **different** photos for testing than for training.
Photograph the faces of 2 or 3 scrambled states of your cube under different lighting, label them like above, and run this cell.
I have not been able to test this step on your real photos, so treat it as an experiment and compare the numbers before and after.

In [ ]:
# file name -> 9 letters (w y r o b g), row by row, as seen in the standard cube net
REAL_TRAIN = {
    # "scramble1_U.jpg": "wrbowgyob",
}
REAL_TEST = {
    # "scramble3_U.jpg": "wrbowgyob",
}

def load_real(d):
    faces, labels = [], []
    for name, letters in d.items():
        face, _ = extract_face(cv2.imread(name), 96)
        faces.append(face[..., ::-1].copy()); labels.append([LETTER[c] for c in letters])
    return np.stack(faces), np.array(labels).reshape(-1, 3, 3)

def augment(img, rng):
    x = img.astype(np.float32) * rng.uniform(0.7, 1.3) + rng.uniform(-25, 25)
    x = np.clip(x + np.random.normal(0, rng.uniform(0, 6), x.shape), 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(x, (0, 0), rng.uniform(0.3, 1.2)) if rng.random() < 0.4 else x

if REAL_TRAIN:
    Xr, yr = load_real(REAL_TRAIN); rng = random.Random(0)
    Xa = np.stack([augment(Xr[i % len(Xr)], rng) for i in range(len(Xr) * 100)])
    ya = np.concatenate([yr] * 100)
    sel = np.random.permutation(len(X_train))[:8000]                    # keep some synthetic data
    Xmix = np.concatenate([Xa, X_train[sel]]); ymix = np.concatenate([ya, y_train[sel]])
    p = np.random.permutation(len(Xmix)); Xmix, ymix = Xmix[p], ymix[p]
    if REAL_TEST:
        Xt, yt = load_real(REAL_TEST)
        print("real test sticker accuracy BEFORE:", (model.predict(Xt, verbose=0).argmax(-1) == yt).mean())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(Xmix, ymix, epochs=5, batch_size=64)
    if REAL_TEST:
        print("real test sticker accuracy AFTER: ", (model.predict(Xt, verbose=0).argmax(-1) == yt).mean())
    model.save("face_cnn_finetuned.keras")

## Step 12: Run the web app from Colab (optional)
Upload `app.py`, `cube_utils.py` and `face_cnn.keras` to the Colab file panel first. Colab has no public address, so the app is exposed through a tunnel; the browser needs HTTPS to use the webcam, which the tunnel provides. I could not test this step inside Colab, so if it fails, run the app on your own computer or deploy it to Streamlit Community Cloud or Hugging Face Spaces (see README).

In [ ]:
!pip install -q streamlit
!npm install -q localtunnel
!streamlit run app.py --server.port 8501 &>/content/logs.txt &
import urllib; print("Tunnel password (IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx localtunnel --port 8501